# Ray Data Preprocessing with Checkpointing and Suspension Handling

In this notebook, we will demonstrate:
 * Ray Data preprocessing with automatic checkpointing
 * RayJob suspension/resume handling
 * Preprocessing resume from checkpoints
 * Both existing cluster and lifecycled cluster scenarios
 * Ray Dashboard integration for monitoring


## Import Required Packages

First, let's import the necessary CodeFlare SDK packages and Ray Data dependencies.


In [ ]:
from codeflare_sdk import (
    Cluster, 
    ClusterConfiguration, 
    RayJob, 
    TokenAuthentication
)

## Ray Data Preprocessing Script

The Ray Data preprocessing script with checkpointing is available as a separate Python file (`ray_data_preprocessing_with_checkpointing.py`). This script demonstrates distributed data preprocessing with fault tolerance.


## Authentication

Set up authentication for accessing the cluster resources.


In [ ]:
# Create authentication object
auth = TokenAuthentication(
    token="XXXXX",  # Replace with your actual token
    server="XXXXX",  # Replace with your actual server URL
    skip_tls=False
)
auth.login()


## Cluster Configuration for Ray Data Processing

Let's create a cluster configuration optimized for Ray Data preprocessing workloads.


In [ ]:
# Define Ray Data preprocessing resource requirements
print("=== Ray Data Preprocessing Cluster Configuration ===")

# Ray Data preprocessing requires more memory and CPU for distributed processing
preprocessing_config = ClusterConfiguration(
    name='ray-data-preprocessing-cluster',
    num_workers=4,  # More workers for distributed processing
    head_cpu_requests='2',
    head_cpu_limits='2',
    head_memory_requests=8,
    head_memory_limits=8,
    worker_cpu_requests='4',  # Higher CPU for data processing
    worker_cpu_limits='4',
    worker_memory_requests=16,  # Higher memory for large datasets
    worker_memory_limits=16,
    # No GPU needed for data preprocessing
    worker_extended_resource_requests={},
    worker_extended_resource_limits={}
)

print("\n📊 Cluster Configuration:")
print(f"  Name: {preprocessing_config.name}")
print(f"  Workers: {preprocessing_config.num_workers}")
print(f"  Head CPU: {preprocessing_config.head_cpu_requests}")
print(f"  Head Memory: {preprocessing_config.head_memory_requests}GB")
print(f"  Worker CPU: {preprocessing_config.worker_cpu_requests}")
print(f"  Worker Memory: {preprocessing_config.worker_memory_requests}GB")
print("\n💡 This configuration is optimized for:")
print("  - Distributed data processing with Ray Data")
print("  - Large memory requirements for datasets")
print("  - High CPU utilization for transformations")
print("  - Fault-tolerant batch processing")


In [ ]:
# Create existing cluster for Ray Data preprocessing
print("=== Creating Existing Cluster for Ray Data Preprocessing ===")

# Use the preprocessing configuration
cluster = Cluster(preprocessing_config)
cluster.apply()

print("\n🚀 Cluster submitted! Waiting for it to be ready...")
cluster.wait_ready()

print("\n✅ Cluster is ready! Status:")
cluster.status()


In [ ]:
# Submit Ray Data preprocessing RayJob to existing cluster
print("=== Submitting Ray Data Preprocessing RayJob to Existing Cluster ===")

# Create RayJob with Ray Data preprocessing script
rayjob_existing = RayJob(
    job_name="ray-data-preprocessing-existing",
    cluster_name="ray-data-preprocessing-cluster",
    namespace="default",
    entrypoint="python ray_data_preprocessing_with_checkpointing.py",
    runtime_env={
        "pip": ["ray[data]>=2.8.0", "pandas", "numpy", "pyarrow"],
        "env_vars": {
            "RAY_DISABLE_IMPORT_WARNING": "1"
        }
    },
    shutdown_after_job_finishes=False,  # Keep cluster running
    ttl_seconds_after_finished=300
)

print("\n📋 RayJob Configuration:")
print(f"  Job name: ray-data-preprocessing-existing")
print(f"  Cluster: ray-data-preprocessing-cluster")
print(f"  Entrypoint: python ray_data_preprocessing_with_checkpointing.py")
print(f"  Runtime environment: Ray Data, pandas, numpy, pyarrow")

# Submit the job
print("\n🚀 Submitting RayJob...")
submission_result = rayjob_existing.submit()
print(f"RayJob submitted successfully: {submission_result}")


## Suspension Testing for Ray Data Preprocessing

Let's demonstrate how to test suspension and resume with Ray Data preprocessing.


In [ ]:
# Demonstrate suspension handling for Ray Data preprocessing
print("=== Suspension Testing for Ray Data Preprocessing ===")

# Create a RayJob for suspension testing
suspension_test_job = RayJob(
    job_name="ray-data-preprocessing-suspension-test",
    entrypoint="python ray_data_preprocessing_with_checkpointing.py",
    cluster_config=preprocessing_config,
    namespace="default",
    runtime_env={
        "pip": ["ray[data]>=2.8.0", "pandas", "numpy", "pyarrow"],
        "env_vars": {
            "RAY_DISABLE_IMPORT_WARNING": "1"
        }
    },
    shutdown_after_job_finishes=True,
    ttl_seconds_after_finished=300
)

print("\n🚀 Submitting Ray Data preprocessing job for suspension testing...")
submission_result = suspension_test_job.submit()
print(f"Job submitted: {submission_result}")

print("\n📊 Monitor job status manually:")
print("  status, ready = suspension_test_job.status()")
print("  print(f'Status: {status}, Ready: {ready}')")

print("\n💡 The Ray Data preprocessing script includes:")
print("  - Ray actor-based checkpoint management")
print("  - Batch processing with checkpoint saving")
print("  - Automatic resume from latest checkpoint")
print("  - Distributed processing across multiple workers")
print("  - Ready for suspension scenarios")


## Manual Suspension Testing

To test how the Ray Data preprocessing works with suspension, you can manually suspend the RayJob.
The following steps demonstrate how to test suspension and checkpointing with Ray Data:
1. Wait for the job to start running (check status)
2. Manually suspend the RayJob using kubectl or oc commands
3. Check job status - it should show SUSPENDED
4. Resume the job using kubectl or oc commands
5. The preprocessing will resume from the latest checkpoint

## Executable Commands for Suspension Testing

The following cells contain the actual commands you can run to test suspension and resume.


In [ ]:
# Check job status before suspension
status, ready = suspension_test_job.status()
print(f"Job status before suspension: {status}")
print(f"Job ready: {ready}")


### Suspend the RayJob

Run one of these commands in your terminal to suspend the job:


In [ ]:
!oc patch rayjob pytorch-training-preemptible -p '{\"spec\":{\"suspend\":true}}'

In [ ]:
# Check job status after suspension
status, ready = suspension_test_job.status()
print(f"Job status after suspension: {status}")
print(f"Job ready: {ready}")
print(f"Expected: SUSPENDED")


### Resume the RayJob

Run one of these commands in your terminal to resume the job:


In [ ]:
!oc patch rayjob pytorch-training-preemptible -p '{\"spec\":{\"suspend\":false}}'


In [ ]:
# Check job status after resume
status, ready = suspension_test_job.status()
print(f"Job status after resume: {status}")
print(f"Job ready: {ready}")
print(f"Expected: RUNNING (processing will resume from checkpoint)")


In [ ]:
# Check RayJob logs to see checkpoint loading
import subprocess

print("Checking RayJob logs for checkpoint loading messages...")
print("💡 Tip: The Ray Dashboard (see next section) provides a better interface for viewing logs!")

# Get the RayJob pod name first
try:
    # Get the pod name for the RayJob
    pod_result = subprocess.run([
        "oc", "get", "pods", "-l", "ray.io/job-name=ray-data-preprocessing-suspension-test", 
        "-o", "jsonpath={.items[0].metadata.name}"
    ], capture_output=True, text=True, check=True)
    
    pod_name = pod_result.stdout.strip()
    if pod_name:
        print(f"Found RayJob pod: {pod_name}")
        
        # Get the logs
        log_result = subprocess.run([
            "oc", "logs", pod_name, "--tail=20"
        ], capture_output=True, text=True, check=True)
        
        print("\n📋 Recent RayJob logs:")
        print("=" * 50)
        print(log_result.stdout)
        print("=" * 50)
        
        # Look for checkpoint messages
        if "Checkpoint loaded" in log_result.stdout:
            print("\n✅ Found checkpoint loading message!")
        elif "Batch" in log_result.stdout or "Processing" in log_result.stdout:
            print("\n📊 Found preprocessing progress messages")
        else:
            print("\n💡 No checkpoint messages found yet - job may still be starting")
            
    else:
        print("No RayJob pod found yet")
        
except subprocess.CalledProcessError as e:
    print(f"Failed to get logs: {e.stderr}")
    print("You can check logs manually with:")
    print("oc get pods -l ray.io/job-name=ray-data-preprocessing-suspension-test")
    print("oc logs <pod-name>")
    print("\n🌐 Or use the Ray Dashboard for a better log viewing experience!")


## Ray Dashboard for Better Log Monitoring

The Ray Dashboard provides a much better interface for monitoring your Ray jobs, viewing logs, and tracking progress. Let's get the dashboard URL and show you how to access it.


In [ ]:
# Get Ray Dashboard URL using CodeFlare SDK
print("Getting Ray Dashboard URL using CodeFlare SDK...")

try:
    # Use the native CodeFlare SDK function to get dashboard URI
    dashboard_url = cluster.cluster_dashboard_uri()
    
    if dashboard_url:
        print(f"\n🌐 Ray Dashboard Access:")
        print(f"   {dashboard_url}")
    else:
        print("No dashboard URL found - cluster may not be ready yet")
        
except Exception as e:
    print(f"Failed to get dashboard URL: {e}")
    print("Make sure the cluster is ready and running")


## Manual Job Status Monitoring

You can check the status of your Ray Data preprocessing jobs manually using the `.status()` method.


In [ ]:
# Example of manual job status monitoring
print("=== Manual Job Status Monitoring ===")

jobs = [
    ("Existing Cluster Job", rayjob_existing),
    ("Lifecycled Cluster Job", rayjob_lifecycled),
    ("Suspension Test Job", suspension_test_job)
]

print("\n📊 Check job status manually:")
print("Example usage:")
print("  status, ready = rayjob_existing.status()")
print("  print(f'Status: {status}, Ready: {ready}')")

print("\n💡 Status Explanations:")
print("  RUNNING    : Job is actively processing data")
print("  SUSPENDED  : Job has been preempted/paused")
print("  COMPLETE   : Job finished successfully")
print("  FAILED     : Job encountered an error")
print("  UNKNOWN    : Status cannot be determined")

print("\n🔄 Run the status check manually when you want to check progress.")
print("💡 For Ray Data preprocessing, look for batch processing progress in logs.")


## Cleanup

Finally, let's clean up our resources.


In [ ]:
# Clean up resources
print("=== Cleaning Up Resources ===")

# Take down the existing cluster
print("Taking down existing cluster...")
cluster.down()

print("\n✅ Cleanup completed!")

print("\n📋 Summary of what we demonstrated:")
print("  ✅ Ray Data for distributed preprocessing")
print("  ✅ Ray actor-based checkpoint management")
print("  ✅ RayJob submission to existing cluster")
print("  ✅ RayJob with lifecycled cluster")
print("  ✅ Suspension/resume capabilities")
print("  ✅ Large-scale data processing with fault tolerance")
print("  ✅ Job status monitoring and tracking")

print("\n📁 Files created:")
print("  - ray_data_preprocessing_with_checkpointing.py (reusable preprocessing script)")
print("  - Synthetic datasets in /tmp/ray_data_input/")
print("  - Processed data in /tmp/ray_data_output/")
print("  - Checkpoints in /tmp/ray_preprocessing_checkpoints/")


## Conclusion

This notebook demonstrated comprehensive Ray Data preprocessing with the CodeFlare SDK:

### **Key Features Demonstrated:**

1. **Ray Data for Distributed Preprocessing**:
   - Distributed data processing across multiple workers
   - Ray Data transformations and batch processing
   - Large-scale dataset handling

2. **Ray Actor-Based Checkpointing**:
   - Ray actor for managing preprocessing state
   - Batch-level checkpoint saving
   - Automatic checkpoint loading and resume

3. **Fault-Tolerant Processing**:
   - Suspension/resume capabilities
   - Progress tracking across batches
   - Error handling and recovery

4. **Multiple Deployment Scenarios**:
   - Existing cluster: For persistent preprocessing infrastructure
   - Lifecycled cluster: For ephemeral preprocessing jobs
   - Both scenarios support suspension handling

### **Best Practices Shown:**

- **Distributed Processing**: Ray Data for scalable data transformations
- **Checkpointing**: Ray actor-based state management
- **Resource Optimization**: Memory and CPU configuration for data processing
- **Error Handling**: Graceful handling of suspension and errors
- **Monitoring**: Comprehensive job status tracking
- **Cleanup**: Proper resource cleanup after processing

### **Ray Data Preprocessing Benefits:**

- **Scalability**: Distributed processing across multiple workers
- **Fault Tolerance**: Checkpointing enables resume after suspension
- **Performance**: Optimized for large-scale data processing
- **Flexibility**: Works with existing and lifecycled clusters
- **Production-Ready**: Handles real-world suspension scenarios

This provides a robust foundation for production Ray Data preprocessing workloads that can handle suspension and resource constraints gracefully.
